# Notebook 04 — DNA Replication and Repair

**Module:** 07 — Biochemistry and Molecular Biology  
**Notebook:** 04 of 10  
**Prerequisites:** Module 05 NB01–NB03, Module 06 NB03 (mutation types)  
**Time estimate:** 60 minutes

> **NGS connection:** DNA polymerase error rates and repair mechanisms are the
> biological origin of sequencing errors and mutation signatures. Understanding
> replication mechanics is required to interpret quality scores (Module 09).

---
## Step 1 — Motivation

Every next-generation sequencing read is produced by a modified DNA polymerase. The
error profile, read length, and coverage uniformity all trace back to the biochemistry
of DNA synthesis and repair. This notebook covers the mechanics of replication and
repair at the level needed to understand why sequencing works the way it does.

---
## Step 3 — Biological Background

### Replication fork machinery

| Component | Function |
|-----------|----------|
| Helicase | Unwinds the double helix; breaks H-bonds between strands |
| Primase | Synthesises short RNA primer (~10 nt) to prime DNA synthesis |
| DNA Pol III (prokaryotes) / Pol δ/ε (eukaryotes) | Adds dNTPs 5'→3'; requires primer |
| DNA Pol I (prokaryotes) / FEN1 (eukaryotes) | Removes RNA primer; fills the gap |
| DNA Ligase | Seals the nick between adjacent Okazaki fragments |
| SSB proteins / RPA | Stabilise single-stranded DNA ahead of the fork |

### Leading vs. lagging strand

DNA synthesis is only possible 5'→3'. At the replication fork:
- **Leading strand:** continuous synthesis in the direction of fork movement
- **Lagging strand:** discontinuous synthesis as **Okazaki fragments** (~200 bp in
  eukaryotes, ~1-2 kb in bacteria); each requires a new primer

**NGS relevance:** Illumina paired-end reads often have higher error rates at the
3' end because the sequencing polymerase, like the cellular enzyme, is less accurate
without the stabilising context of the preceding bases.

### DNA Polymerase fidelity

| Mechanism | Error rate (per base copied) |
|-----------|-----------------------------|
| Base selection only | ~10⁻⁵ |
| + 3'→5' proofreading exonuclease | ~10⁻⁷ |
| + Mismatch repair (MMR) | ~10⁻⁹ to 10⁻¹⁰ |

Human genome ≈ 6×10⁹ base pairs. At 10⁻⁹ error rate: ~6 errors per genome
duplication. This is why somatic mutation rates are low (but not zero).

### Repair pathways

| Pathway | Substrate | Disease when defective |
|---------|-----------|------------------------|
| Base Excision Repair (BER) | Oxidised/deaminated bases, AP sites | Accumulation of 8-oxoG, C→T transitions |
| Nucleotide Excision Repair (NER) | Bulky adducts, UV dimers (CPDs) | Xeroderma pigmentosum (XP) |
| Mismatch Repair (MMR) | Replication mismatches | Lynch syndrome (hereditary CRC) |
| Homologous Recombination (HR) | Double-strand breaks | BRCA1/2 mutations → breast/ovarian cancer |
| Non-homologous End Joining (NHEJ) | Double-strand breaks (G1 phase) | Generates small indels |

**COSMIC mutation signatures:**
- SBS1 (clock): C→T at CpG — spontaneous deamination of 5mC, accumulates with age
- SBS2/13 (APOBEC): C→T and C→G at TpC — in NER-deficient or APOBEC-overexpressing tumours
- SBS6/15 (MMR deficiency): enriched indels at microsatellites (microsatellite instability, MSI)

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import log10

In [ ]:
# Cell 6.1 — DNA polymerase fidelity calculation
def genome_errors_per_replication(genome_size: int, error_rate: float) -> float:
    """Expected number of errors per genome replication."""
    return genome_size * error_rate

def prob_error_free_gene(gene_length: int, error_rate: float) -> float:
    """Probability that a gene is copied error-free."""
    return (1 - error_rate) ** gene_length

HUMAN_GENOME = 6e9  # haploid * 2
E_COLI_GENOME = 4.6e6

print("DNA Polymerase fidelity — expected errors per replication:")
print(f"{'System':>25} {'Error rate':>12} {'Errors/genome':>16} {'Error-free gene (1 kb)':>24}")
print("-" * 80)
for name, genome, rate in [
    ('E. coli (no proofreading)', E_COLI_GENOME, 1e-5),
    ('E. coli (with proofreading)', E_COLI_GENOME, 1e-7),
    ('E. coli (+ MMR)', E_COLI_GENOME, 1e-9),
    ('Human (somatic)', HUMAN_GENOME, 1e-9),
    ('Human (germline de novo)', HUMAN_GENOME, 1e-10),
]:
    errs = genome_errors_per_replication(int(genome), rate)
    p_free = prob_error_free_gene(1000, rate)
    print(f"  {name:>23} {rate:>12.0e} {errs:>16.1f} {p_free:>24.6f}")

In [ ]:
# Cell 6.2 — Okazaki fragment simulation
def simulate_lagging_strand(template_length: int,
                             okazaki_size: int = 200,
                             primer_length: int = 10,
                             seed: int = 42) -> list:
    """
    Simulate lagging strand synthesis as Okazaki fragments.
    Returns list of (start, end, has_primer) tuples for each fragment.
    """
    rng = np.random.default_rng(seed)
    fragments = []
    pos = template_length

    while pos > 0:
        # Primer laid down, then fragment synthesised rightward
        fragment_length = rng.integers(okazaki_size - 50, okazaki_size + 50)
        start = max(0, pos - fragment_length)
        fragments.append({
            'start': start,
            'end': pos,
            'primer_start': start,
            'primer_end': start + primer_length,
            'length': pos - start,
        })
        pos = start

    return list(reversed(fragments))

frags = simulate_lagging_strand(3000)
print(f"Lagging strand simulation (3000 bp, ~200 bp Okazaki fragments):")
print(f"  Number of Okazaki fragments: {len(frags)}")
print(f"  Average fragment length: {np.mean([f['length'] for f in frags]):.0f} bp")
print(f"  Total RNA primer bases: {len(frags) * 10} bp ({len(frags)*10/3000*100:.1f}% of template)")
print("  [Each primer must be removed and replaced with DNA before ligation]")

In [ ]:
# Cell 6.3 — Mutation accumulation with and without repair
def simulate_mutation_accumulation(n_generations: int, genome_size: int,
                                    error_rate_raw: float, repair_efficiency: float,
                                    seed: int = 42) -> np.ndarray:
    """
    Simulate cumulative mutations over generations.
    repair_efficiency: fraction of errors corrected each generation.
    Returns: cumulative mutation count per generation.
    """
    rng = np.random.default_rng(seed)
    effective_rate = error_rate_raw * (1 - repair_efficiency)
    # Each generation: Poisson-distributed new mutations
    new_muts = rng.poisson(effective_rate * genome_size, size=n_generations)
    return np.cumsum(new_muts)

n_gen = 50
scenarios = [
    ('No repair', 1e-5, 0.0, 'tomato'),
    ('Proofreading only', 1e-5, 0.99, 'orange'),
    ('Proofreading + MMR', 1e-5, 0.9999, 'steelblue'),
]
print("Cumulative mutations after 50 generations (E. coli, 4.6 Mb genome):")
for name, rate, repair, _ in scenarios:
    muts = simulate_mutation_accumulation(n_gen, int(E_COLI_GENOME), rate, repair)
    print(f"  {name:>25}: {muts[-1]:>8,} total mutations")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Okazaki fragment diagram + mutation accumulation curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Panel 1: Okazaki fragment diagram
ax = axes[0]
template_len = 3000
frags_display = frags[:8]  # show first 8 fragments

ax.axhline(0.5, color='steelblue', lw=3, label='Leading strand (continuous)')
ax.text(50, 0.55, "5'→3' →→→", fontsize=8, color='steelblue')

for i, f in enumerate(frags_display):
    y = -0.2
    # DNA portion
    ax.barh(y, f['end'] - f['primer_end'], left=f['primer_end'],
            height=0.15, color='steelblue', alpha=0.7)
    # RNA primer
    ax.barh(y, f['primer_end'] - f['primer_start'], left=f['primer_start'],
            height=0.15, color='tomato', alpha=0.9,
            label='RNA primer' if i == 0 else '')
    # Arrow showing 5'→3' direction
    ax.annotate('', xy=(f['start'] + 5, y), xytext=(f['end'] - 5, y),
                arrowprops=dict(arrowstyle='<-', color='gray', lw=1))

ax.text(50, -0.28, "←←← 5'→3' synthesis (right to left = lagging)", fontsize=8, color='gray')
ax.set_xlim(0, max(f['end'] for f in frags_display) + 100)
ax.set_ylim(-0.5, 0.8)
ax.set_xlabel('Position (bp)')
ax.set_title('Replication fork:\nLeading (continuous) vs. Lagging (Okazaki fragments)')
ax.legend(fontsize=8, loc='upper right')
ax.axis('off')
ax.set_xticks([]); ax.set_yticks([])

# Panel 2: Mutation accumulation
ax = axes[1]
for name, rate, repair, color in scenarios:
    muts = simulate_mutation_accumulation(n_gen, int(E_COLI_GENOME), rate, repair)
    ax.plot(range(n_gen), muts, color=color, lw=2, label=name)

ax.set_xlabel('Generations')
ax.set_ylabel('Cumulative mutations')
ax.set_title('Effect of repair on mutation accumulation\n(E. coli, 4.6 Mb, raw error rate 10⁻⁵)')
ax.legend(fontsize=8)
ax.set_yscale('log')

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. A human somatic cell replicates with error rate 10⁻⁹ per base. How many mutations
   do you expect per cell division in the 6×10⁹ bp human genome? Is this consistent
   with the observed ~1–2 somatic mutations per cell division in normal tissue?
2. Lynch syndrome patients lack functional MMR. If MMR corrects 99% of replication
   errors, how does the somatic mutation rate change in Lynch syndrome tumours?
3. An Illumina sequencer shows elevated error rates in positions 75–100 of 100 bp
   reads. Which phase of the read (corresponding to which strand synthesis mechanism)
   is most likely causing this, and why?
4. CRISPR-Cas9 creates a double-strand break. In G1 phase (no sister chromatid),
   which repair pathway dominates, and what consequence does this have for the edit?

---
## Quiz — Active Recall

1. Why does the lagging strand require Okazaki fragments?
2. What is the error rate of DNA polymerase without proofreading? After proofreading?
3. Which repair pathway is mutated in Lynch syndrome? What type of mutations does it repair?
4. What is an Okazaki fragment? Approximately how long are they in human cells?
5. SBS1 (the 'clock' mutation signature) is caused by what biochemical mechanism?

---
## Reflection

**Date completed:** ____________________

> *[A whole-genome sequencing pipeline reports ~90 somatic mutations in a normal
> blood cell from a 60-year-old. Is this consistent with what you know about
> replication fidelity and the SBS1 signature?]*

---
*Next: `05_transcription_and_translation_mechanics.ipynb`*